# XaitkExplainable Analytics Store Demo

This notebook demonstrates the end-to-end workflow for the XaitkExplainable capability with the analytics store:

1. Construct simulated saliency records (IC and OD)
2. Write to the analytics store
3. Query results via SQL

## Setup

In [ ]:
import tempfile

from checkmaite.core._common.xaitk_explainable_capability import XaitkExplainableRecord
from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend

## Simulated Saliency Records

`XaitkExplainableRun.extract()` produces one record per saliency map:

- **IC runs**: one record per image, with a ground-truth label and saliency statistics
- **OD runs**: one record per detection, with confidence, predicted label, and saliency statistics

We construct records directly here to demonstrate the schema without running a full capability.

In [ ]:
# IC records — 3 images
run_uid_ic = "a1b2c3d4e5f6" * 5  # 60-char placeholder UID

ic_records = [
    XaitkExplainableRecord(
        run_uid=run_uid_ic,
        dataset_id="cifar10",
        model_id="resnet50",
        saliency_generator_type="RISEStack",
        image_index=0,
        gt_label="cat",
        mean_saliency=0.15,
        max_saliency=0.95,
        std_saliency=0.12,
        positive_saliency_ratio=0.45,
    ),
    XaitkExplainableRecord(
        run_uid=run_uid_ic,
        dataset_id="cifar10",
        model_id="resnet50",
        saliency_generator_type="RISEStack",
        image_index=1,
        gt_label="dog",
        mean_saliency=0.22,
        max_saliency=0.88,
        std_saliency=0.18,
        positive_saliency_ratio=0.61,
    ),
    XaitkExplainableRecord(
        run_uid=run_uid_ic,
        dataset_id="cifar10",
        model_id="resnet50",
        saliency_generator_type="RISEStack",
        image_index=2,
        gt_label="car",
        mean_saliency=0.09,
        max_saliency=0.72,
        std_saliency=0.08,
        positive_saliency_ratio=0.31,
    ),
]

# OD records — 2 images, multiple detections each
run_uid_od = "f6e5d4c3b2a1" * 5  # 60-char placeholder UID

od_records = [
    XaitkExplainableRecord(
        run_uid=run_uid_od,
        dataset_id="coco-val",
        model_id="yolov5",
        saliency_generator_type="DRISEStack",
        image_index=0,
        image_id="img_001",
        detection_index=0,
        predicted_label="car",
        confidence=0.92,
        mean_saliency=0.20,
        max_saliency=0.98,
        std_saliency=0.18,
        positive_saliency_ratio=0.55,
    ),
    XaitkExplainableRecord(
        run_uid=run_uid_od,
        dataset_id="coco-val",
        model_id="yolov5",
        saliency_generator_type="DRISEStack",
        image_index=0,
        image_id="img_001",
        detection_index=1,
        predicted_label="person",
        confidence=0.78,
        mean_saliency=0.11,
        max_saliency=0.65,
        std_saliency=0.10,
        positive_saliency_ratio=0.38,
    ),
    XaitkExplainableRecord(
        run_uid=run_uid_od,
        dataset_id="coco-val",
        model_id="yolov5",
        saliency_generator_type="DRISEStack",
        image_index=1,
        image_id="img_002",
        detection_index=0,
        predicted_label="truck",
        confidence=0.55,
        mean_saliency=0.07,
        max_saliency=0.50,
        std_saliency=0.06,
        positive_saliency_ratio=0.22,
    ),
]

print(f"IC records: {len(ic_records)}")
print(f"OD records: {len(od_records)}")

## Write to Analytics Store

In [ ]:
store_dir = tempfile.mkdtemp(prefix="xaitk_demo_")
backend = ParquetBackend(store_dir)
store = AnalyticsStore(backend)

# Write all records directly
backend.write(ic_records)
backend.write(od_records)

print(f"Store path: {store_dir}")
print(f"Tables: {store.list_tables()}")
print(f"\nSchema: {store.describe_table('xaitk_explainable')}")

## Query via SQL

In [ ]:
# Reconstruct per-image saliency stats (IC)
results_df = store.query_sql("""
    SELECT
        image_index,
        gt_label,
        mean_saliency,
        max_saliency,
        std_saliency,
        positive_saliency_ratio
    FROM xaitk_explainable
    WHERE gt_label IS NOT NULL
    ORDER BY image_index
""")
print(results_df)

In [ ]:
# Filter OD detections by confidence + saliency
results_df = store.query_sql("""
    SELECT
        image_id,
        detection_index,
        predicted_label,
        confidence,
        mean_saliency,
        positive_saliency_ratio
    FROM xaitk_explainable
    WHERE confidence IS NOT NULL
      AND confidence >= 0.7
      AND positive_saliency_ratio >= 0.4
    ORDER BY confidence DESC
""")
print(results_df)

In [ ]:
# Summary stats grouped by saliency generator type
results_df = store.query_sql("""
    SELECT
        saliency_generator_type,
        COUNT(*) AS record_count,
        AVG(mean_saliency) AS avg_mean_saliency,
        AVG(positive_saliency_ratio) AS avg_positive_ratio
    FROM xaitk_explainable
    GROUP BY saliency_generator_type
""")
print(results_df)

## Cross-Capability JOIN Example

When other capabilities (e.g., `DataevalCleaning`) have been run on the same dataset,
you can correlate saliency quality with dataset quality metrics:

```sql
-- Find images with weak saliency in datasets with high outlier ratios
SELECT
    x.image_index,
    x.mean_saliency,
    x.positive_saliency_ratio,
    c.image_outlier_ratio
FROM xaitk_explainable x
JOIN dataeval_cleaning c ON x.dataset_id = c.dataset_id
WHERE x.positive_saliency_ratio < 0.2
```